# 第12回：改善実験を小さく回す

**今日の問い：改善した理由を後から説明できる実験とは何か。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X, y = df[features], df["active"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## TRY：1要素だけ変えて記録する


In [ ]:
rows=[]
for depth in [3, 6, None]:
    model=make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=depth, random_state=42))
    scores=cross_validate(model, X, y, cv=cv, scoring="f1", return_train_score=True)
    rows.append({"実験名": f"depth={depth}", "変更点": "max_depthのみ", "学習F1": scores["train_score"].mean(), "検証F1平均": scores["test_score"].mean(), "検証F1標準偏差": scores["test_score"].std()})
experiment_log=pd.DataFrame(rows)
experiment_log.round(3)


## 重要度から次の仮説を考える


In [ ]:
best=make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)).fit(X, y)
importance=permutation_importance(best, X, y, scoring="f1", n_repeats=10, random_state=42)
pd.DataFrame({"特徴量": features, "重要度": importance.importances_mean}).sort_values("重要度", ascending=False).round(3)


## 実験ログの最小項目

- 実験名
- 変えたもの（1つ）
- 固定した比較条件
- 結果の平均とばらつき
- 気づき
- 次の仮説

Copilotには案を出してもらい、優先順位と予測時点の妥当性は人が判断します。
